# Spark SQL Cheatsheet

Quick reference for Spark SQL operations and optimization.

## Table of Contents
- [Basic SQL Operations](#basic-sql-operations)
- [Advanced SQL Features](#advanced-sql-features)
- [Performance Optimization](#performance-optimization)
- [Catalog Operations](#catalog-operations)
- [Delta Lake Operations](#delta-lake-operations)

## Basic SQL Operations

In [ ]:
# Register DataFrame as temp view
df.createOrReplaceTempView("employees")
df.createGlobalTempView("global_employees")

# Run SQL query
result = spark.sql("SELECT * FROM employees")
result.show()

# Basic SELECT
spark.sql("SELECT name, age FROM employees")
spark.sql("SELECT * FROM employees WHERE age > 25")

# ORDER BY
spark.sql("SELECT * FROM employees ORDER BY age DESC")
spark.sql("SELECT * FROM employees ORDER BY age ASC, name ASC")

# LIMIT
spark.sql("SELECT * FROM employees LIMIT 10")

# DISTINCT
spark.sql("SELECT DISTINCT department FROM employees")

## Aggregations

In [ ]:
# GROUP BY
spark.sql("SELECT department, COUNT(*) FROM employees GROUP BY department")
spark.sql("SELECT department, AVG(salary) FROM employees GROUP BY department")

# HAVING
spark.sql("""
    SELECT department, COUNT(*) as cnt 
    FROM employees 
    GROUP BY department 
    HAVING cnt > 5
""")

# Multiple aggregations
spark.sql("""
    SELECT department, 
           COUNT(*) as count,
           SUM(salary) as total,
           AVG(salary) as average,
           MAX(salary) as maximum,
           MIN(salary) as minimum
    FROM employees 
    GROUP BY department
""")

## Joins in SQL

In [ ]:
# Inner join
spark.sql("""
    SELECT a.*, b.department 
    FROM employees a 
    INNER JOIN departments b ON a.dept_id = b.id
""")

# Left join
spark.sql("""
    SELECT a.*, b.department 
    FROM employees a 
    LEFT JOIN departments b ON a.dept_id = b.id
""")

# Right join
spark.sql("""
    SELECT a.*, b.department 
    FROM employees a 
    RIGHT JOIN departments b ON a.dept_id = b.id
""")

# Full outer join
spark.sql("""
    SELECT a.*, b.department 
    FROM employees a 
    FULL OUTER JOIN departments b ON a.dept_id = b.id
""")

# Self join
spark.sql("""
    SELECT a.name as employee, b.name as manager
    FROM employees a 
    LEFT JOIN employees b ON a.manager_id = b.id
""")

## Advanced SQL Features

In [ ]:
# Window functions
spark.sql("""
    SELECT name, department, salary,
           ROW_NUMBER() OVER (PARTITION BY department ORDER BY salary DESC) as rank
    FROM employees
""")

# CTE (Common Table Expression)
spark.sql("""
    WITH dept_stats AS (
        SELECT department, AVG(salary) as avg_salary
        FROM employees
        GROUP BY department
    )
    SELECT e.*, d.avg_salary
    FROM employees e
    JOIN dept_stats d ON e.department = d.department
""")

# CASE statement
spark.sql("""
    SELECT name, salary,
           CASE 
               WHEN salary < 50000 THEN 'Low'
               WHEN salary < 80000 THEN 'Medium'
               ELSE 'High'
           END as salary_level
    FROM employees
""")

# Subqueries
spark.sql("""
    SELECT * FROM employees 
    WHERE salary > (SELECT AVG(salary) FROM employees)
""")

## Performance Optimization

In [ ]:
# Query hints
spark.sql("""
    SELECT /*+ BROADCAST(departments) */ a.*, b.department
    FROM employees a 
    JOIN departments b ON a.dept_id = b.id
""")

# Repartition hint
spark.sql("""
    SELECT /*+ REPARTITION(10) */ * FROM employees
""")

# Cache table
spark.sql("CACHE TABLE employees")
spark.sql("UNCACHE TABLE employees")

# Check if table is cached
spark.sql("SHOW TABLES").show()

# Explain plan
spark.sql("EXPLAIN SELECT * FROM employees").show()
spark.sql("EXPLAIN EXTENDED SELECT * FROM employees").show()

## Catalog Operations

In [ ]:
# List databases
spark.sql("SHOW DATABASES")
spark.sql("SHOW SCHEMAS")

# Use database
spark.sql("USE my_database")

# Create database
spark.sql("CREATE DATABASE IF NOT EXISTS my_database")

# Drop database
spark.sql("DROP DATABASE IF EXISTS my_database")

# List tables
spark.sql("SHOW TABLES")
spark.sql("SHOW TABLES IN my_database")

# Describe table
spark.sql("DESCRIBE employees")
spark.sql("DESCRIBE EXTENDED employees")

# Create table
spark.sql("""
    CREATE TABLE IF NOT EXISTS employees (
        id INT,
        name STRING,
        age INT,
        salary DOUBLE
    )
    USING PARQUET
""")

# Drop table
spark.sql("DROP TABLE IF EXISTS employees")

## Delta Lake Operations

In [ ]:
# Create Delta table
spark.sql("""
    CREATE TABLE IF NOT EXISTS employees_delta (
        id INT,
        name STRING,
        age INT,
        salary DOUBLE
    )
    USING DELTA
""")

# Write to Delta table
df.write.format("delta").mode("overwrite").saveAsTable("employees_delta")

# Time travel
spark.sql("SELECT * FROM employees_delta VERSION AS OF 0")
spark.sql("SELECT * FROM employees_delta TIMESTAMP AS OF '2024-01-01'")

# Vacuum old versions
spark.sql("VACUUM employees_delta RETAIN 168 HOURS")

# Update Delta table
spark.sql("UPDATE employees_delta SET salary = salary * 1.1 WHERE age > 30")

# Delete from Delta table
spark.sql("DELETE FROM employees_delta WHERE age < 25")

# Merge into Delta table
spark.sql("""
    MERGE INTO employees_delta target
    USING updates source
    ON target.id = source.id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

## Configuration Tips

- **spark.sql.shuffle.partitions**: Controls shuffle partition count (default 200)
- **spark.sql.adaptive.enabled**: Enable adaptive query execution (default true)
- **spark.sql.autoBroadcastJoinThreshold**: Broadcast join threshold (default 10MB)
- **spark.sql.inMemoryColumnarStorage.compressed**: Enable compression for cached data
- **spark.sql.codegen.wholeStage**: Enable whole-stage code generation